# Clustered outages

GENCO is evaluated on the released test sets, where the lines out are spread over the whole grid. Storms and wildfires
take out lines in one area. This notebook builds such outages and compares GENCO on them with random outages of the same
size.

- Real N-2 operating points from the released test set, with 8 or 18 more lines removed: close together around a random
  centre bus (clustered), or anywhere (random, the control). Each new case is solved with lightsim2grid.
- GENCO's error, the decisions it misses, and what a Newton check from its state recovers.
- Timing of lightsim2grid and of compiled float16 GENCO on the same machine.

Runtime: T4 GPU, about 2 GB of downloads, 30 to 50 minutes. Output: `clustered_outages.zip` (unzip into `results/`).

## Settings

In [ ]:
REPO_ID = "gridfm/genco-pf-contingency"
HF_REVISION = "5c625b00f5b79c8a7d4d27ecc133d4ecdf573af7"       # the revision every run used (unchanged since 1 October 2026)
GRAPHKIT_COMMIT = "95b90eecc7f5c2c86a9dd776f0c3f40434def22f"   # the commit every run used (branch head from 1 to 14 October 2026)
LOCAL_DIR = "genco-pf-contingency"
OUT_DIR = "clustered_outages"
GRAPHKIT_DIR = "gridfm-graphkit"
GRAPHKIT_BRANCH = "genco-paper-repro"
NETWORK = "Texas2k_case1_2016summerpeak"
MODEL_DIR = "model/training_bs_16"

LEVELS = [20]                 # released N-20 scenarios: known hard cases + random ones
N_NORMAL = 100
COMPARE_LEVEL = 20
BASE_LEVEL, BASE_PARTS = 2, [0, 1]   # real N-2 operating points used to build the storm-like cases
N_BASE = 60                   # operating points; each gets clustered and uniform outages at two levels
STORM_TOTALS = [10, 20]       # total outages after adding lines to the 2 already out
BATCH = 8
TIME_BATCH = 32
N_TIMING = 40                 # scenarios used for the solver timing
TOL, TOL_SCREEN, MAX_NR = 1e-8, 1e-2, 6
BAD_VM = 0.01
N_DEMO = {"random": 6, "hard": 6, "clustered": 6, "uniform": 3}
MAX_DOWNLOAD_GB = 5
SEED = 0

In [ ]:
# [GPU] install and import

import importlib.util, os, subprocess, sys, time


def run(cmd):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:]); print(r.stderr[-3000:])
        raise RuntimeError(f"Command failed: {cmd}")


import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all again.\n"
          "The notebook also works on CPU, but the timing results would not be meaningful.")

if not os.path.isdir(GRAPHKIT_DIR):
    run(f"git clone -q -b {GRAPHKIT_BRANCH} https://github.com/gridfm/gridfm-graphkit.git {GRAPHKIT_DIR}")
    run(f"git -C {GRAPHKIT_DIR} checkout -q {GRAPHKIT_COMMIT}")   # pinned: the commit every run used
need = {"torch_geometric": "torch_geometric", "lightning": "lightning", "pytorch_lightning": "pytorch-lightning",
        "huggingface_hub": "huggingface_hub", "pyarrow": "pyarrow", "seaborn": "seaborn", "sklearn": "scikit-learn"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
if missing:
    run(f"{sys.executable} -m pip install -q " + " ".join(missing))

import torch_geometric   # imported before the replacement below, so PyTorch Geometric keeps its own native code paths
USED_SHIM = importlib.util.find_spec("torch_scatter") is None
if USED_SHIM:
    shim = os.path.abspath(os.path.join("shims", "torch_scatter"))
    os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "__init__.py"), "w") as f:
        f.write('"""Small pure-PyTorch stand-in for torch_scatter (only the functions gridfm-graphkit uses)."""\nimport torch\n\n__version__ = "0.0.0-shim"\n\n\ndef _prep(src, index, dim):\n    dim = dim if dim >= 0 else src.dim() + dim\n    if index.dim() != src.dim():\n        shape = [1] * src.dim()\n        shape[dim] = -1\n        index = index.view(shape).expand_as(src)\n    return index, dim\n\n\ndef _size(src, index, dim, dim_size):\n    size = list(src.shape)\n    if dim_size is None:\n        dim_size = int(index.max()) + 1 if index.numel() > 0 else 0\n    size[dim] = dim_size\n    return size\n\n\ndef scatter_add(src, index, dim=-1, out=None, dim_size=None):\n    index, dim = _prep(src, index, dim)\n    if out is None:\n        out = torch.zeros(_size(src, index, dim, dim_size), dtype=src.dtype, device=src.device)\n    return out.scatter_add_(dim, index, src)\n\n\nscatter_sum = scatter_add\n\n\ndef scatter_mean(src, index, dim=-1, out=None, dim_size=None):\n    total = scatter_add(src, index, dim, None, dim_size)\n    idx, d = _prep(src, index, dim)\n    count = torch.zeros_like(total).scatter_add_(d, idx, torch.ones_like(src)).clamp_(min=1)\n    return total / count\n\n\ndef scatter_max(src, index, dim=-1, out=None, dim_size=None):\n    idx, d = _prep(src, index, dim)\n    res = torch.full(_size(src, idx, d, dim_size), float("-inf"), dtype=src.dtype, device=src.device)\n    res = res.scatter_reduce(d, idx, src, reduce="amax", include_self=True)\n    res[torch.isinf(res)] = 0\n    return res, None\n\n\ndef scatter(src, index, dim=-1, out=None, dim_size=None, reduce="sum"):\n    if reduce in ("sum", "add"):\n        return scatter_add(src, index, dim, out, dim_size)\n    if reduce == "mean":\n        return scatter_mean(src, index, dim, out, dim_size)\n    if reduce == "max":\n        return scatter_max(src, index, dim, out, dim_size)[0]\n    raise NotImplementedError(reduce)\n')
    sys.path.insert(0, os.path.abspath("shims"))
    print("torch_scatter is not installed: using a small built-in replacement")
sys.path.insert(0, os.path.abspath(GRAPHKIT_DIR))

import gridfm_graphkit
from gridfm_graphkit.io.param_handler import NestedNamespace, load_normalizer, get_task_transforms, get_task
from gridfm_graphkit.training.callbacks import adapt_state_dict_for_model
from gridfm_graphkit.datasets.hetero_preprocess import build_hetero_data_for_scenario, merge_agg_gen_into_bus
from gridfm_graphkit.datasets.globals import VM_OUT, VA_OUT
from torch_geometric.loader import DataLoader
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch_geometric", torch_geometric.__version__, "| gridfm-graphkit from", os.path.abspath(GRAPHKIT_DIR), "| device:", DEVICE)

In [ ]:
# lightsim2grid: open-source C++ Newton-Raphson (KLU sparse solver), used here as the professional AC solver
import importlib.util, subprocess, sys
if importlib.util.find_spec("lightsim2grid") is None:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightsim2grid"], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
print("lightsim2grid installed:", importlib.util.find_spec("lightsim2grid") is not None)

In [ ]:
# ===================== core functions =====================
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import splu

SN_MVA = 100.0
BUS_COLS = ["scenario", "bus", "Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF",
            "min_vm_pu", "max_vm_pu", "GS", "BS", "Va_dc", "vn_kv"]
BR_COLS = ["scenario", "idx", "from_bus", "to_bus", "Yff_r", "Yff_i", "Yft_r", "Yft_i", "Ytf_r", "Ytf_i",
           "Ytt_r", "Ytt_i", "rate_a", "br_status", "pf_dc", "pt_dc"]
PRED_ALIASES = {"vm": ["vm_pu", "Vm_pred", "VM_pred", "vm_pred"], "va": ["va", "Va_pred", "VA_pred", "va_pred"]}


# ---------------------------------------------------------------- file helpers
def find_local(local_dir, k, table):
    base = Path(local_dir) / "data" / f"test_{k}"
    hits = sorted(base.rglob(f"{table}.parquet")) if base.exists() else []
    if not hits:
        raise FileNotFoundError(f"No {table}.parquet under {base}")
    return hits[0]


def find_pred(local_dir, k):
    base = Path(local_dir) / "mlflow" / "eval" / f"test_{k}"
    hits = sorted(base.rglob("*predictions.parquet")) if base.exists() else []
    hits = [h for h in hits if not re.search(r"_(gen|branch)_predictions\.parquet$", h.name)]
    if not hits:
        raise FileNotFoundError(f"No predictions.parquet under {base}")
    return hits[0]


def _dataset(path):
    return ds.dataset(str(path), format="parquet", partitioning="hive")


def read_cols(path, wanted, scen=None):
    dset = _dataset(path)
    cols = [c for c in wanted if c in dset.schema.names]
    flt = None
    if scen is not None:
        values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
        flt = ds.field("scenario").isin(values)
    return dset.to_table(columns=cols, filter=flt).to_pandas()


def scenario_ids(path):
    return np.unique(_dataset(path).to_table(columns=["scenario"]).column("scenario").to_numpy().astype(np.int64))


# ---------------------------------------------------------------- one chunk of scenarios
def load_chunk(bus_path, br_path, pred_chunk, scen):
    """Arrays for a chunk of scenarios. pred_chunk: DataFrame with scenario, bus, vm, va (already filtered)."""
    bus = read_cols(bus_path, BUS_COLS, scen=scen)
    bus["scenario"] = bus["scenario"].astype(np.int64)
    bus = bus.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    pred = pred_chunk.sort_values(["scenario", "bus"], kind="stable").reset_index(drop=True)
    S = len(scen)
    nb = len(bus) // S
    if len(bus) != S * nb or len(pred) != S * nb:
        raise RuntimeError(f"rows do not match: bus {len(bus)}, predictions {len(pred)}, {S} scenarios")
    ids = np.broadcast_to(np.arange(nb), (S, nb))
    if not (np.array_equal(bus["bus"].to_numpy().reshape(S, nb), ids) and np.array_equal(pred["bus"].to_numpy().reshape(S, nb), ids)):
        raise RuntimeError("bus numbers are not 0..nb-1 in every scenario")
    A = {c: bus[c].to_numpy(np.float64).reshape(S, nb) for c in BUS_COLS[2:] if c in bus.columns}
    A["PQ"], A["PV"], A["REF"] = A["PQ"] > 0.5, A["PV"] > 0.5, A["REF"] > 0.5
    A["Vm_g"] = pred["vm"].to_numpy(np.float64).reshape(S, nb)
    va = pred["va"].to_numpy(np.float64).reshape(S, nb)
    rad_err = np.nanmedian(np.abs(np.degrees(va) - A["Va"]))
    deg_err = np.nanmedian(np.abs(va - A["Va"]))
    A["va_unit"] = "rad" if rad_err <= deg_err else "deg"
    A["Va_g"] = np.degrees(va) if A["va_unit"] == "rad" else va
    br = read_cols(br_path, BR_COLS, scen=scen)
    A["sc"] = np.searchsorted(scen, br["scenario"].to_numpy().astype(np.int64))
    A["line"] = (br["idx"].to_numpy().astype(np.int64) if "idx" in br.columns
                 else br.groupby("scenario").cumcount().to_numpy().astype(np.int64))
    A["f"] = br["from_bus"].to_numpy().astype(np.int64)
    A["t"] = br["to_bus"].to_numpy().astype(np.int64)
    for y in ["Yff", "Yft", "Ytf", "Ytt"]:
        A[y] = br[y + "_r"].to_numpy(np.float64) + 1j * br[y + "_i"].to_numpy(np.float64)
    A["rate"] = br["rate_a"].to_numpy(np.float64)
    A["on"] = (br["br_status"].to_numpy(np.float64) > 0.5) if "br_status" in br.columns else (np.abs(A["Yff"]) > 0)
    pdc = br["pf_dc"].to_numpy(np.float64) if "pf_dc" in br.columns else np.full(len(br), np.nan)
    ptdc = br["pt_dc"].to_numpy(np.float64) if "pt_dc" in br.columns else pdc
    A["P_dc_abs"] = np.maximum(np.abs(pdc), np.abs(ptdc))
    A["S"], A["nb"], A["scen"] = S, nb, np.asarray(scen)
    return A


def branch_flows(A, V):
    sc, f, t = A["sc"], A["f"], A["t"]
    Vf, Vt = V[sc, f], V[sc, t]
    Sf = Vf * np.conj(A["Yff"] * Vf + A["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(A["Ytt"] * Vt + A["Ytf"] * Vf) * SN_MVA
    return Sf, St


def bus_mismatch(A, Vm, Sf, St):
    S, nb, sc, f, t = A["S"], A["nb"], A["sc"], A["f"], A["t"]
    add = lambda v, at: np.bincount(sc * nb + at, weights=v, minlength=S * nb).reshape(S, nb)
    P = (A["Pg"] - A["Pd"]) - (add(Sf.real, f) + add(St.real, t)) - A["GS"] * Vm**2 * SN_MVA
    Q = (A["Qg"] - A["Qd"]) - (add(Sf.imag, f) + add(St.imag, t)) + A["BS"] * Vm**2 * SN_MVA
    return P, Q


def _scen_max(values, sc, mask, S):
    out = np.full(S, np.nan)
    v = np.where(mask, values, -np.inf)
    tmp = np.full(S, -np.inf); np.maximum.at(tmp, sc, v)
    out[np.isfinite(tmp)] = tmp[np.isfinite(tmp)]
    return out


def chunk_tables(A, near_line=0.4, near_v=0.08):
    """Compact line-level and bus-level tables plus per-scenario signals for one chunk."""
    S, nb = A["S"], A["nb"]
    V_t = A["Vm"] * np.exp(1j * np.radians(A["Va"]))
    V_g = A["Vm_g"] * np.exp(1j * np.radians(A["Va_g"]))
    V_d = np.exp(1j * np.radians(A["Va_dc"]))
    Sf_t, St_t = branch_flows(A, V_t)
    Sf_g, St_g = branch_flows(A, V_g)
    Sf_d, St_d = branch_flows(A, V_d)
    P_t, _ = bus_mismatch(A, A["Vm"], Sf_t, St_t)
    P_g, Q_g = bus_mismatch(A, A["Vm_g"], Sf_g, St_g)
    resP = np.where(A["REF"], np.nan, np.abs(P_g))
    with np.errstate(all="ignore"):
        r_p99 = np.nanpercentile(resP, 99, axis=1)
    rated = A["on"] & (A["rate"] > 0)
    rate = np.where(rated, A["rate"], np.nan)
    ld = lambda Sf, St: np.maximum(np.abs(Sf), np.abs(St)) / rate
    ld_t, ld_g, ld_dS = ld(Sf_t, St_t), ld(Sf_g, St_g), ld(Sf_d, St_d)
    ld_dP = A["P_dc_abs"] / rate
    dc_failed = np.isnan(A["Va_dc"]).any(axis=1)
    keep = rated & ((ld_t >= near_line) | (ld_g >= near_line) | (np.nan_to_num(ld_dS) >= near_line) | (np.nan_to_num(ld_dP) >= near_line))
    lines = pd.DataFrame({
        "sidx": A["sc"][keep].astype(np.int32), "line": A["line"][keep].astype(np.int32),
        "ld_true": ld_t[keep].astype(np.float32), "ld_genco": ld_g[keep].astype(np.float32),
        "ld_dc_paper": ld_dS[keep].astype(np.float32), "ld_dc_p": ld_dP[keep].astype(np.float32),
    })
    lines.loc[dc_failed[lines.sidx.to_numpy()], ["ld_dc_paper", "ld_dc_p"]] = np.nan
    # PQ buses close to a voltage limit (truth or GENCO)
    PQ, vmin, vmax = A["PQ"], A["min_vm_pu"], A["max_vm_pu"]
    dist_t = np.minimum(A["Vm"] - vmin, vmax - A["Vm"])
    dist_g = np.minimum(A["Vm_g"] - vmin, vmax - A["Vm_g"])
    kb = PQ & ((dist_t <= near_v) | (dist_g <= near_v))
    si, bi = np.nonzero(kb)
    buses = pd.DataFrame({"sidx": si.astype(np.int32), "bus": bi.astype(np.int32),
                          "vm_true": A["Vm"][kb].astype(np.float32), "vm_genco": A["Vm_g"][kb].astype(np.float32),
                          "vmin": vmin[kb].astype(np.float32), "vmax": vmax[kb].astype(np.float32)})
    scen = pd.DataFrame({
        "scenario": A["scen"], "res_p99": r_p99,
        "n_lines_out": np.bincount(A["sc"][~A["on"]], minlength=S),
        "n_overload_true": np.bincount(A["sc"][rated], weights=(ld_t[rated] >= 1.0), minlength=S),
        "n_vviol_true": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax))).sum(axis=1),
        "err_vm_max": np.nanmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), np.nan), axis=1),
        "worst_bus": np.nanargmax(np.where(PQ, np.abs(A["Vm_g"] - A["Vm"]), -1.0), axis=1),
        "err_load_max": _scen_max(np.abs(ld_g - ld_t), A["sc"], rated, S),
        "n_missed_overload_genco": np.bincount(A["sc"][rated], weights=((ld_t[rated] >= 1.0) & (ld_g[rated] < 1.0)), minlength=S),
        "n_missed_vviol_genco": (PQ & ((A["Vm"] < vmin) | (A["Vm"] > vmax)) & (A["Vm_g"] >= vmin) & (A["Vm_g"] <= vmax)).sum(axis=1),
        "dc_failed": dc_failed,
    })
    scen["worst_bus_kv"] = A["vn_kv"][np.arange(S), scen["worst_bus"].to_numpy()] if "vn_kv" in A else np.nan
    checks = {"truth_mismatch_max_MW": float(np.nanmax(np.abs(np.where(A["REF"], 0, P_t)))),
              "va_unit": A["va_unit"]}
    return lines, buses, scen, checks


# ---------------------------------------------------------------- margins
def fit_exponent(e, r, n_bins=8, q=0.99, min_count=200):
    """Slope of log(quantile of error) against log(scenario residual), from bins of the residual. Clipped to [0, 2]."""
    e, r = np.asarray(e, float), np.asarray(r, float)
    ok = np.isfinite(e) & np.isfinite(r) & (r > 0)
    e, r = e[ok], r[ok]
    if len(e) < 3 * min_count:
        return 0.0, []
    if len(e) < n_bins * min_count:
        n_bins = max(3, len(e) // min_count)
    edges = np.unique(np.quantile(r, np.linspace(0, 1, n_bins + 1)))
    idx = np.clip(np.searchsorted(edges, r, side="right") - 1, 0, len(edges) - 2)
    xs, ys = [], []
    for b in range(len(edges) - 1):
        m = idx == b
        if m.sum() >= min_count:
            qq = np.quantile(e[m], q)
            if qq > 0:
                xs.append(np.log(np.median(r[m]))); ys.append(np.log(qq))
    if len(xs) < 3:
        return 0.0, list(zip(xs, ys))
    return float(np.clip(np.polyfit(xs, ys, 1)[0], 0.0, 2.0)), list(zip(xs, ys))


def scale_of(r, r_ref, b):
    return np.power(np.maximum(np.asarray(r, float), 1e-9) / r_ref, b)


# ---------------------------------------------------------------- screening curves
def screening_curve(score, positive, sidx_global, n_scen, thresholds):
    """Flag an item if score >= threshold. Returns recall, false alarms per scenario, share of scenarios with a flag."""
    score = np.asarray(score, float); positive = np.asarray(positive, bool)
    ok = np.isfinite(score)
    score, positive, sidx = score[ok], positive[ok], np.asarray(sidx_global)[ok]
    n_pos = max(int(positive.sum()), 1)
    order = np.argsort(-score, kind="stable")
    s_desc = -score[order]                          # ascending
    cum_pos = np.r_[0, np.cumsum(positive[order])]
    cum_neg = np.r_[0, np.cumsum(~positive[order])]
    smax = np.full(n_scen, -np.inf); np.maximum.at(smax, sidx, score)
    smax_sorted = np.sort(smax)
    th = np.asarray(thresholds, float)
    n = np.searchsorted(s_desc, -th, side="right")   # items with score >= th
    tp, fp = cum_pos[n], cum_neg[n]
    sent = n_scen - np.searchsorted(smax_sorted, th, side="left")
    return pd.DataFrame({"threshold": th, "recall": tp / n_pos, "misses": n_pos - tp,
                         "false_alarms_per_scenario": fp / n_scen, "share_scenarios_flagged": sent / n_scen})


# ---------------------------------------------------------------- Newton-Raphson (polar, MATPOWER formulation)
def build_ybus(nb, f, t, Yff, Yft, Ytf, Ytt, GS, BS):
    rows = np.concatenate([f, f, t, t, np.arange(nb)])
    cols = np.concatenate([f, t, f, t, np.arange(nb)])
    vals = np.concatenate([Yff, Yft, Ytf, Ytt, GS + 1j * BS])
    return sp.csr_matrix((vals, (rows, cols)), shape=(nb, nb))


def dsbus_dv(Ybus, V):
    Ibus = Ybus @ V
    dV, dI, dVn = sp.diags(V), sp.diags(Ibus), sp.diags(V / np.abs(V))
    dS_dVm = dV @ (Ybus @ dVn).conj() + dI.conj() @ dVn
    dS_dVa = 1j * dV @ (dI - Ybus @ dV).conj()
    return dS_dVm.tocsr(), dS_dVa.tocsr()


def newton(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=20, n_steps=None):
    """Returns final V, iterations done, list of max mismatch (p.u.) after each iteration, first-step correction size."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]
    n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V); norms = [float(np.max(np.abs(F)))]; it = 0; first_step = np.nan
    while True:
        if n_steps is None and (norms[-1] <= tol or it >= max_it):
            break
        if n_steps is not None and it >= n_steps:
            break
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            norms.append(np.inf); break
        if it == 0:
            first_step = float(np.max(np.abs(dx)))
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V); it += 1; norms.append(float(np.max(np.abs(F))))
        if not np.isfinite(norms[-1]) or norms[-1] > 1e6:
            break
    return V, it, norms, first_step


def scenario_system(A, i):
    """Ybus, specified power and bus sets for scenario i of a chunk."""
    nb = A["nb"]
    m = (A["sc"] == i) & A["on"]
    Y = build_ybus(nb, A["f"][m], A["t"][m], A["Yff"][m], A["Yft"][m], A["Ytf"][m], A["Ytt"][m], A["GS"][i], A["BS"][i])
    Sspec = (A["Pg"][i] - A["Pd"][i] + 1j * (A["Qg"][i] - A["Qd"][i])) / SN_MVA
    pv, pq = np.nonzero(A["PV"][i])[0], np.nonzero(A["PQ"][i])[0]
    return Y, Sspec, pv, pq


def extract_scenario(A, i):
    """Keep only what Newton needs for scenario i (small, so it can be stored)."""
    m = A["sc"] == i
    out = {key: A[key][i].copy() for key in ["Pd", "Qd", "Pg", "Qg", "Vm", "Va", "PQ", "PV", "REF", "GS", "BS",
                                             "Va_dc", "Vm_g", "Va_g", "min_vm_pu", "max_vm_pu", "vn_kv"] if key in A}
    for key in ["f", "t", "Yff", "Yft", "Ytf", "Ytt", "rate", "on", "line"]:
        if key in A:
            out[key] = A[key][m].copy()
    out["nb"] = A["nb"]; out["scenario"] = int(A["scen"][i])
    return out


def run_newton_cases(d, tol=1e-8, max_it=20):
    """Flat start, DC start, GENCO start (to convergence) and GENCO + exactly one step, for one stored scenario."""
    nb = d["nb"]
    on = d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq, ref = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0], np.nonzero(d["REF"])[0]
    Vm_set = np.where(d["PQ"], 1.0, d["Vm"])          # setpoints at PV and slack, 1.0 elsewhere
    Va_ref = np.where(d["REF"], d["Va"], 0.0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def start(Vm, Va_deg):
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(Vm_set, Va_ref), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    res = {"scenario": d["scenario"]}
    for name, V0 in starts.items():
        t0 = time.perf_counter()
        V, it, norms, step1 = newton(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        res[f"seconds_{name}"] = time.perf_counter() - t0
        conv = norms[-1] <= tol
        res[f"iters_{name}"] = it if conv else np.nan
        res[f"converged_{name}"] = bool(conv)
        res[f"start_mismatch_{name}"] = norms[0]
        if conv:
            res[f"max_dV_vs_truth_{name}"] = float(np.max(np.abs(V - V_true)))
        if name == "genco":
            res["first_step_size"] = step1
    t0 = time.perf_counter()
    V1, _, norms1, _ = newton(Y, Sspec, starts["genco"], pv, pq, n_steps=1)
    res["seconds_genco_1step"] = time.perf_counter() - t0
    res["mismatch_after_1_step"] = norms1[-1]
    rated = on & (d["rate"] > 0)

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))
    for name, V in [("genco", starts["genco"]), ("genco_1step", V1)]:
        ld = loading(V); Vm = np.abs(V)
        vio = d["PQ"] & ((Vm < d["min_vm_pu"]) | (Vm > d["max_vm_pu"]))
        res[f"err_vm_max_{name}"] = float(np.max(np.abs(Vm - d["Vm"])[d["PQ"]])) if d["PQ"].any() else np.nan
        res[f"err_load_max_{name}"] = float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan
        res[f"missed_overloads_{name}"] = int((over_t & (ld < 1.0)).sum())
        res[f"missed_vviol_{name}"] = int((vio_t & ~vio).sum())
    res["true_overloads"] = int(over_t.sum()); res["true_vviol"] = int(vio_t.sum())
    return res


def newton_trace(Ybus, Sspec, V0, pv, pq, tol=1e-8, max_it=10):
    """List of (V, max mismatch p.u., seconds for this step) after 0, 1, 2, ... Newton steps."""
    V = V0.astype(complex).copy()
    Va, Vm = np.angle(V), np.abs(V)
    pvpq = np.r_[pv, pq]; n1 = len(pvpq)

    def mis(V):
        S = V * np.conj(Ybus @ V) - Sspec
        return np.r_[S[pvpq].real, S[pq].imag]

    F = mis(V)
    out = [(V.copy(), float(np.max(np.abs(F))), 0.0)]
    for _ in range(max_it):
        if out[-1][1] <= tol:
            break
        t0 = time.perf_counter()
        dS_dVm, dS_dVa = dsbus_dv(Ybus, V)
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])], format="csc")
        try:
            dx = -splu(J).solve(F)
        except RuntimeError:
            break
        Va[pvpq] += dx[:n1]; Vm[pq] += dx[n1:]
        V = Vm * np.exp(1j * Va)
        F = mis(V)
        out.append((V.copy(), float(np.max(np.abs(F))), time.perf_counter() - t0))
        if not np.isfinite(out[-1][1]) or out[-1][1] > 1e6:
            break
    return out


def run_trace_cases(d, tol=1e-8, max_it=10, line_margin=0.0, v_margin=0.0):
    """Decision metrics after every Newton step from flat, DC and GENCO starts, for one stored scenario.
    Columns ending in _m use the safety margins (flag a line from 1 - line_margin, a bus within v_margin of a limit)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    pv, pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
    PQ, lo, hi = d["PQ"], d["min_vm_pu"], d["max_vm_pu"]
    rated = on & (d["rate"] > 0)
    V_true = d["Vm"] * np.exp(1j * np.radians(d["Va"]))

    def loading(V):
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[rated] / d["rate"][rated]

    ld_t = loading(V_true)
    over_t = ld_t >= 1.0
    vio_t = PQ & ((d["Vm"] < lo) | (d["Vm"] > hi))

    def start(Vm, Va_deg):
        Vm = np.where(PQ, Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    starts = {"flat": start(np.ones(nb), np.zeros(nb)), "dc": start(np.ones(nb), np.nan_to_num(d["Va_dc"])),
              "genco": start(d["Vm_g"], d["Va_g"])}
    rows = []
    for name, V0 in starts.items():
        trace = newton_trace(Y, Sspec, V0, pv, pq, tol=tol, max_it=max_it)
        converged = trace[-1][1] <= tol
        for j, (V, mism, secs) in enumerate(trace):
            ld, Vm = loading(V), np.abs(V)
            vio = PQ & ((Vm < lo) | (Vm > hi))
            vio_m = PQ & ((Vm < lo + v_margin) | (Vm > hi - v_margin))
            fl_m = ld >= 1.0 - line_margin
            rows.append({"scenario": d["scenario"], "start": name, "steps": j, "mismatch": mism, "seconds_step": secs,
                         "converged_at_end": converged, "iterations_to_converge": len(trace) - 1 if converged else np.nan,
                         "err_vm_max": float(np.max(np.abs(Vm - d["Vm"])[PQ])) if PQ.any() else np.nan,
                         "err_load_max": float(np.max(np.abs(ld - ld_t))) if len(ld) else np.nan,
                         "missed_overloads": int((over_t & (ld < 1.0)).sum()), "false_overloads": int((~over_t & (ld >= 1.0)).sum()),
                         "missed_vviol": int((vio_t & ~vio).sum()), "false_vviol": int((~vio_t & vio).sum()),
                         "missed_overloads_m": int((over_t & ~fl_m).sum()), "false_overloads_m": int((~over_t & fl_m).sum()),
                         "missed_vviol_m": int((vio_t & ~vio_m).sum()), "false_vviol_m": int((~vio_t & vio_m).sum()),
                         "true_overloads": int(over_t.sum()), "true_vviol": int(vio_t.sum())})
    return rows


# ===================== GENCO helpers =====================
from scipy.sparse.csgraph import dijkstra


class ScenarioEval:
    """Truth-based checks of any voltage state for one stored scenario (from extract_scenario)."""

    def __init__(self, d):
        self.d = d
        nb, on = d["nb"], d["on"]
        self.Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
        self.Sspec = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
        self.pv, self.pq = np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]
        self.nonref = np.nonzero(~d["REF"])[0]
        self.rated = on & (d["rate"] > 0)
        Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
        self.ld_t = self.loading(Vt)
        self.over_t = self.ld_t >= 1.0
        self.vio_t = d["PQ"] & ((d["Vm"] < d["min_vm_pu"]) | (d["Vm"] > d["max_vm_pu"]))

    def loading(self, V):
        d = self.d
        Vf, Vt = V[d["f"]], V[d["t"]]
        Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
        St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
        return np.maximum(np.abs(Sf), np.abs(St))[self.rated] / d["rate"][self.rated]

    def state(self, Vm, Va_deg):
        d = self.d
        Vm = np.where(d["PQ"], Vm, d["Vm"]); Va = np.where(d["REF"], d["Va"], Va_deg)
        return Vm * np.exp(1j * np.radians(Va))

    def metrics(self, Vm, Va_deg):
        d = self.d
        V = self.state(Vm, Va_deg)
        S = V * np.conj(self.Y @ V) - self.Sspec
        Pm, Qm = np.abs(S.real), np.abs(S.imag)
        ld = self.loading(V)
        Vabs = np.abs(V)
        vio = d["PQ"] & ((Vabs < d["min_vm_pu"]) | (Vabs > d["max_vm_pu"]))
        err = np.abs(Vabs - d["Vm"])
        return {"err_vm_max": float(err[d["PQ"]].max()) if d["PQ"].any() else np.nan,
                "err_load_max": float(np.max(np.abs(ld - self.ld_t))) if len(ld) else np.nan,
                "mismatch_pu": float(max(Pm[np.r_[self.pv, self.pq]].max(), Qm[self.pq].max())),
                "res_p99_MW": float(np.percentile(Pm[self.nonref], 99) * SN_MVA),
                "missed_overloads": int((self.over_t & (ld < 1.0)).sum()),
                "missed_vviol": int((self.vio_t & ~vio).sum())}

    def bus_errors(self, Vm):
        return np.where(self.d["PQ"], np.abs(Vm - self.d["Vm"]), np.nan)


def thevenin_pq(d, block=200):
    """|Z_ii| for every PQ bus with PV and slack buses held at fixed voltage: how much the voltage at a bus
    moves per unit of current injected there (large = electrically weak spot)."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    pq = np.nonzero(d["PQ"])[0]
    lu = splu(Y[pq][:, pq].tocsc())
    n = len(pq); z = np.empty(n, complex)
    for s in range(0, n, block):
        idx = np.arange(s, min(n, s + block))
        E = np.zeros((n, len(idx)), complex); E[idx, np.arange(len(idx))] = 1.0
        X = lu.solve(E)
        z[idx] = X[idx, np.arange(len(idx))]
    return pq, np.abs(z)


def hops_to_voltage_control(d):
    """Number of in-service lines between each bus and the nearest PV or slack bus."""
    nb, on = d["nb"], d["on"]
    A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
    src = np.nonzero(d["PV"] | d["REF"])[0]
    dist = dijkstra(A, directed=False, unweighted=True, indices=src, min_only=True)
    return dist


def dc_solve_seconds(d, reps=3):
    """Time of one DC power flow (one sparse solve on the susceptance matrix), for the cost comparison."""
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    keep = np.nonzero(~d["REF"])[0]
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter()
    for _ in range(reps):
        B = (-Y.imag)[keep][:, keep].tocsc()
        splu(B).solve(P)
    return (time.perf_counter() - t0) / reps


# ===================== outage construction =====================
from scipy.sparse.csgraph import connected_components
try:
    from lightsim2grid.algorithm import NR_KLU as _KLU
    HAVE_KLU = True
except Exception:
    try:
        from lightsim2grid.solver import KLUSolver as _KLU
        HAVE_KLU = True
    except Exception:
        HAVE_KLU = False


def system_of(d):
    nb, on = d["nb"], d["on"]
    Y = build_ybus(nb, d["f"][on], d["t"][on], d["Yff"][on], d["Yft"][on], d["Ytf"][on], d["Ytt"][on], d["GS"], d["BS"])
    S = (d["Pg"] - d["Pd"] + 1j * (d["Qg"] - d["Qd"])) / SN_MVA
    return Y, S, np.nonzero(d["REF"])[0], np.nonzero(d["PV"])[0], np.nonzero(d["PQ"])[0]


def klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol=1e-8):
    # lightsim2grid's C++ Newton-Raphson with the KLU sparse solver. Returns V, converged, iterations, seconds.
    w = np.zeros(Y.shape[0]); w[ref] = 1.0 / len(ref)
    Yc = Y.tocsc()
    s = _KLU()
    t0 = time.perf_counter()
    ok = s.solve(Yc, np.asarray(V0, complex), np.asarray(S, complex), ref.astype(np.int32), w,
                 pv.astype(np.int32), pq.astype(np.int32), int(max_iter), float(tol))
    dt = time.perf_counter() - t0
    V = np.array(s.get_V(), dtype=complex, copy=True)
    return V, bool(ok), int(s.get_nb_iter()), dt


def dc_angles(d):
    # DC power flow on the current topology (series susceptances, no shunts, no phase shifts). Returns angles in degrees.
    nb, on = d["nb"], d["on"]
    f, t, b = d["f"][on], d["t"][on], d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[f, t, f, t], np.r_[f, t, t, f])), shape=(nb, nb)).tocsc()
    ref = np.nonzero(d["REF"])[0]
    keep = np.setdiff1d(np.arange(nb), ref)
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    th = np.zeros(nb)
    th[keep] = splu(B[keep][:, keep]).solve(P)
    return np.degrees(th) + float(d["Va"][ref[0]])


def connected(nb, f, t):
    A = sp.csr_matrix((np.ones(len(f)), (f, t)), shape=(nb, nb))
    return connected_components(A, directed=False)[0] == 1


def pick_outages(d, extra, mode, rng):
    # Indices (into the scenario's branch rows) of `extra` more lines to remove, keeping the grid in one piece.
    # mode "clustered": the lines nearest (in hops) to a random centre bus, like a storm footprint.
    # mode "uniform": random lines anywhere, like the released test sets.
    nb = d["nb"]; on = d["on"].copy()
    cand = np.nonzero(on & (d["rate"] > 0))[0]
    hop = None
    if mode == "clustered":
        center = int(rng.integers(nb))
        A = sp.csr_matrix((np.ones(on.sum()), (d["f"][on], d["t"][on])), shape=(nb, nb))
        hop = dijkstra(A, directed=False, unweighted=True, indices=center)
        dist = np.minimum(hop[d["f"][cand]], hop[d["t"][cand]])
        order = cand[np.lexsort((rng.random(len(cand)), dist))]
    else:
        order = rng.permutation(cand)
    removed = []
    for i in order:
        if len(removed) == extra:
            break
        trial = on.copy(); trial[i] = False
        if connected(nb, d["f"][trial], d["t"][trial]):
            on = trial; removed.append(int(i))
    if len(removed) < extra:
        return None, None
    radius = float(np.max(np.minimum(hop[d["f"][removed]], hop[d["t"][removed]]))) if hop is not None else np.nan
    return np.array(removed), radius


def solve_new_topology(d, removed, tol=1e-8, max_iter=30):
    # Exact AC solution after removing `removed`; starts: the base solution, then flat, then DC.
    d2 = {k: (v.copy() if isinstance(v, np.ndarray) else v) for k, v in d.items()}
    d2["on"][removed] = False
    Y, S, ref, pv, pq = system_of(d2)
    base = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    flat = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(np.where(d2["REF"], d2["Va"], 0.0)))
    d2["Va_dc"] = dc_angles(d2)
    dc = np.where(d2["PQ"], 1.0, d2["Vm"]) * np.exp(1j * np.radians(d2["Va_dc"]))
    for name, V0 in [("base", base), ("flat", flat), ("dc", dc)]:
        if HAVE_KLU:
            V, ok, it, _ = klu_solve(Y, S, V0, ref, pv, pq, max_iter, tol)
        else:
            V, it, norms, _ = newton(Y, S, V0, pv, pq, tol=tol, max_it=max_iter)
            ok = norms[-1] <= tol
        if ok:
            d2["Vm"], d2["Va"] = np.abs(V), np.degrees(np.angle(V))
            d2["solved_from"] = name
            return d2
    return None


# ---------------------------------------------------------------- notebook setup
import json, math, platform, zipfile, warnings, copy
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
RNG = np.random.default_rng(SEED)

# Known hard cases (scenario ids per outage level) and how often each bus was the worst bus
HARD = {int(k): v for k, v in json.loads('{"2": [21, 22, 129, 131, 133, 158, 171, 436, 552, 555, 1231, 1251, 1264, 1282, 1294, 1396, 1446, 1449, 1457, 1521, 1574, 1578, 2212, 2222, 2288, 2353, 2390, 2394, 3314, 3379, 3851, 3852, 3900, 3903, 3905, 3927, 3928, 3947, 4622, 4628, 4698, 4711, 4722, 4723, 4762, 4777, 4778, 4779, 4798, 4801, 4851, 4853, 4878, 4896, 4898, 4899, 4902, 4920, 4923, 4924, 4961, 5231, 5258, 5368, 5382, 5949, 6025, 6057, 6208, 6230, 6294, 6319, 6320, 6356, 6367, 7259, 7261, 7356, 7658, 7684, 7689, 7844, 7863, 7870, 7882, 7968, 8188, 8255, 8411, 8487, 8595], "10": [30, 51, 98, 122, 141, 1035, 1047, 1051, 1070, 1080, 1105, 1113, 1148, 1170, 1180, 1420, 1440, 1464, 1466, 1469, 1478, 1566, 1584, 1585, 1595, 1599, 5404, 5440, 5482, 5485, 5509, 5512, 5526, 5531, 5534, 5556, 5563, 5582, 5583, 5584, 7632, 7645, 7667, 7705, 7706, 7711, 7749, 7773], "20": [2210, 2214, 2225, 2231, 2243, 2248, 2249, 2252, 2289, 2296, 2308, 2340, 2352, 2360, 2368, 2383, 2399, 2627, 2650, 2651, 2656, 2658, 2661, 2678, 2680, 2683, 2688, 2693, 2694, 2705, 2717, 2727, 2749, 2761, 2775, 2778, 3607, 3609, 3610, 3641, 3657, 3671, 3690, 3699, 3711, 3722, 3732, 3734, 3746, 3748, 3753, 3758, 3760, 3770, 3773, 3779, 3780, 3782, 3783, 3784, 3787, 3799, 5606, 5608, 5611, 5612, 5620, 5624, 5636, 5644, 5664, 5667, 5677, 5685, 5692, 5704, 5706, 5708, 5710, 5716, 5721, 5726, 5728, 5729, 5742, 5764, 5766, 5784, 5800, 5802, 5804, 5824, 5835, 5860, 5868, 5869, 5890, 5902, 5933, 5938, 5953, 5959, 5978, 5986, 5988, 5992]}').items()}
WORST_BUS_COUNTS = {int(k): v for k, v in json.loads('{"1797": 92, "1869": 29, "1872": 15, "1971": 13, "182": 12, "366": 11, "548": 10, "1669": 9, "137": 8, "1874": 7, "1875": 6, "253": 5, "1845": 5, "583": 4, "926": 4, "1025": 4, "498": 4, "235": 4, "640": 4, "806": 3, "298": 3, "1860": 3, "1985": 3, "751": 3, "1873": 3, "844": 3, "11": 3, "1842": 3, "1880": 3, "369": 3, "1668": 2, "1206": 2, "1266": 2, "517": 2, "367": 2, "106": 2, "88": 1, "1256": 1, "123": 1, "1281": 1, "130": 1, "1877": 1, "1893": 1, "554": 1, "119": 1, "150": 1, "314": 1, "246": 1, "1642": 1, "724": 1, "575": 1, "200": 1, "1275": 1, "1815": 1, "1464": 1, "1465": 1, "662": 1, "355": 1, "124": 1, "1981": 1, "674": 1, "387": 1, "1953": 1, "499": 1, "45": 1, "174": 1, "1853": 1, "346": 1, "1113": 1, "362": 1, "805": 1, "882": 1, "417": 1, "1535": 1, "1407": 1, "222": 1, "1924": 1, "348": 1, "245": 1}').items()}
RAW3_PAT = re.compile(r"(?:^|/)(bus_data|branch_data|gen_data)\.parquet/scenario_partition=(\d+)/[^/]+$")
PART = 200


def read_all(path, scen):
    dset = ds.dataset(str(path), format="parquet", partitioning="hive")
    values = pa.array(np.asarray(scen, dtype=np.int64)).cast(dset.schema.field("scenario").type)
    t = dset.to_table(filter=ds.field("scenario").isin(values)).to_pandas()
    return t.drop(columns=[c for c in ["scenario_partition"] if c in t.columns])

print("ready | hard cases per level:", {k: len(v) for k, v in HARD.items()})

In [ ]:
print("lightsim2grid available to the solver code:", HAVE_KLU)
if not HAVE_KLU:
    print("WARNING: part A will use only the Python solver (see the installation output above).")

## 1. Download: model, N-20 blocks with the hard cases, two N-2 blocks for the storm cases, N-20 predictions

In [ ]:
from huggingface_hub import HfApi, snapshot_download
print("Listing the repository ...")
all_files = [(f.path, f.size) for f in HfApi().list_repo_tree(REPO_ID, recursive=True, revision=HF_REVISION) if getattr(f, "size", None) is not None]
local_has = lambda p: (Path(LOCAL_DIR) / p).exists()
plan = {p: s for p, s in all_files if p.startswith(MODEL_DIR + "/")}
plan.update({p: s for p, s in all_files if p.startswith(f"mlflow/eval/test_{COMPARE_LEVEL}/")
             and (p.endswith("predictions.parquet") or p.endswith("meta.yaml")) and not re.search(r"_(gen|branch)_predictions", p)})
want = {k: {s // PART for s in HARD.get(k, [])} for k in LEVELS}
want.setdefault(BASE_LEVEL, set()).update(BASE_PARTS)
for k, parts in want.items():
    for p, s in all_files:
        m = RAW3_PAT.search(p) if p.startswith(f"data/test_{k}/") else None
        if m and int(m.group(2)) in parts:
            plan[p] = s
new = sum(s for p, s in plan.items() if not local_has(p)) / 1e9
print(f"Planned: {len(plan)} files, {new:.2f} GB not yet on disk")
if new > MAX_DOWNLOAD_GB:
    raise RuntimeError("Planned download is above MAX_DOWNLOAD_GB")
snapshot_download(REPO_ID, allow_patterns=list(plan), local_dir=LOCAL_DIR, revision=HF_REVISION)
print("Download finished")

## 2. Scenarios

In [ ]:
SEL = {}
on_disk = scenario_ids(find_local(LOCAL_DIR, 20, "bus_data"))
hard20 = np.array(sorted(set(HARD.get(20, [])) & set(on_disk.tolist())), dtype=np.int64)
rand20 = RNG.choice(np.setdiff1d(on_disk, hard20), min(N_NORMAL, len(on_disk) - len(hard20)), replace=False)
SEL[20] = np.sort(np.r_[hard20, rand20]).astype(np.int64)
HARD20 = set(hard20.tolist())
base_disk = scenario_ids(find_local(LOCAL_DIR, BASE_LEVEL, "bus_data"))
base_pool = np.setdiff1d(base_disk, np.array(HARD.get(BASE_LEVEL, []), dtype=np.int64))
BASE_IDS = np.sort(RNG.choice(base_pool, min(N_BASE, len(base_pool)), replace=False)).astype(np.int64)
print(f"N-20: {len(hard20)} hard + {len(rand20)} random | storm base points: {len(BASE_IDS)} N-{BASE_LEVEL} scenarios")

In [ ]:
# [GPU] build graphs

import yaml
cfg_path = os.path.join(GRAPHKIT_DIR, "scripts", "contingency", "configs", "caseTexas_eval.yaml")
with open(cfg_path) as f:
    ARGS = NestedNamespace(**yaml.safe_load(f))
stats = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "normalizer_stats.pt"), map_location="cpu", weights_only=True)
stats_net = stats[NETWORK] if NETWORK in stats else (stats if "baseMVA" in stats else next(iter(stats.values())))
NORMALIZER = load_normalizer(ARGS)
NORMALIZER.fit_from_dict(stats_net)
TRANSFORM = get_task_transforms(ARGS)
print("normalizer baseMVA:", NORMALIZER.baseMVA, "| config:", cfg_path)

GRAPHS = {}
for k in LEVELS:
    bus = read_all(find_local(LOCAL_DIR, k, "bus_data"), SEL[k])
    gen = read_all(find_local(LOCAL_DIR, k, "gen_data"), SEL[k])
    br = read_all(find_local(LOCAL_DIR, k, "branch_data"), SEL[k])
    bus = merge_agg_gen_into_bus(bus, gen)
    bg, gg, rg = bus.groupby("scenario"), gen.groupby("scenario"), br.groupby("scenario")
    GRAPHS[k] = []
    for s in SEL[k]:
        b = bg.get_group(s).sort_values("bus").reset_index(drop=True)
        data = build_hetero_data_for_scenario(int(s), b, gg.get_group(s), rg.get_group(s))
        NORMALIZER.transform(data)
        GRAPHS[k].append(TRANSFORM(data))
    print(f"k={k}: {len(GRAPHS[k])} graphs built")
    del bus, gen, br

In [ ]:
# [GPU] load model and smoke test

TASK = get_task(ARGS, [NORMALIZER])
sd = torch.load(os.path.join(LOCAL_DIR, MODEL_DIR, "best_model_state_dict.pt"), map_location="cpu")
sd = adapt_state_dict_for_model(sd, TASK)
missing_keys, unexpected_keys = TASK.load_state_dict(sd, strict=False)
print(f"checkpoint loaded: {len(sd)} tensors | missing {len(missing_keys)} | unexpected {len(unexpected_keys)}")
if missing_keys:
    raise RuntimeError(f"The checkpoint does not match the model. Missing: {missing_keys[:5]}")
MODEL = TASK.model.to(DEVICE).eval()
print("GENCO parameters:", sum(p.numel() for p in MODEL.parameters()))


def forward(model, batch):
    return model(batch.x_dict, batch.edge_index_dict, batch.edge_attr_dict, batch.mask_dict)


test_ids = SEL[COMPARE_LEVEL][:4]
rel = read_cols(find_pred(LOCAL_DIR, COMPARE_LEVEL), ["scenario", "bus", "vm_pu", "va"], scen=test_ids).sort_values(["scenario", "bus"])
with torch.no_grad():
    b = next(iter(DataLoader(GRAPHS[COMPARE_LEVEL][:4], batch_size=4))).to(DEVICE)
    out = forward(MODEL, b)["bus"].float().cpu().numpy()
d_vm = np.abs(out[:, VM_OUT] - rel["vm_pu"].to_numpy())
d_va = np.abs(out[:, VA_OUT] - rel["va"].to_numpy())
print(f"Smoke test vs released predictions (4 scenarios): Vm max diff {d_vm.max():.2e} p.u., Va max diff {d_va.max():.2e} rad")
if np.median(d_vm) > 1e-3:
    raise RuntimeError("Our run does not reproduce the released predictions; check the installation and the downloaded files.")
print("OK: our GENCO run reproduces the released predictions.")

## 3. Run GENCO in float32 and compiled float16

In [ ]:
# [GPU] run genco

try:
    torch._dynamo.reset()
    CMODEL = torch.compile(MODEL, dynamic=True)
    COMPILED = True
except Exception as exc:
    print("torch.compile failed, using the plain model for float16:", repr(exc)[:200]); CMODEL, COMPILED = MODEL, False


def run_genco(graphs, model, amp, bs=BATCH):
    out = {}
    for b in DataLoader(graphs, batch_size=bs, shuffle=False):
        b = b.to(DEVICE)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
            o = forward(model, b)["bus"].float()
        ptr = b["bus"].ptr.cpu().tolist()
        for g, s in enumerate(b["scenario_id"].view(-1).cpu().tolist()):
            out[int(s)] = (o[ptr[g]:ptr[g + 1], VM_OUT].cpu().numpy(), o[ptr[g]:ptr[g + 1], VA_OUT].cpu().numpy())
    return out


def time_model(model, graphs, amp, bs=TIME_BATCH, reps=3):
    batches = [b.to(DEVICE) for b in DataLoader((graphs * 8)[:128], batch_size=bs, shuffle=False)]
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=amp):
        for b in batches[:2]:
            forward(model, b)
        torch.cuda.synchronize(); t0, n = time.perf_counter(), 0
        for _ in range(reps):
            for b in batches:
                forward(model, b); n += b.num_graphs
        torch.cuda.synchronize()
    return 1000 * (time.perf_counter() - t0) / n


GENCO_OUT = {"fp32": run_genco(GRAPHS[20], MODEL, False), "fp16c": run_genco(GRAPHS[20], CMODEL, True)}
T_GENCO = {"fp32": time_model(MODEL, GRAPHS[20], False), "fp16c": time_model(CMODEL, GRAPHS[20], True)}
print("GENCO ms per scenario on this GPU:", {k: round(v, 2) for k, v in T_GENCO.items()}, "| compiled:", COMPILED)

In [ ]:
# Ground truth for the N-20 scenarios and for the storm base points
def truth_dicts(level, ids, preds=None):
    ids = np.sort(ids)
    if preds is None:   # no GENCO answer needed: use the truth itself in the prediction slot
        bus = read_cols(find_local(LOCAL_DIR, level, "bus_data"), ["scenario", "bus", "Vm", "Va"], scen=ids)
        pred = pd.DataFrame({"scenario": bus.scenario.astype(np.int64), "bus": bus.bus.astype(np.int64),
                             "vm": bus.Vm, "va": np.radians(bus.Va)})
    else:
        nbs = [len(preds[int(s)][0]) for s in ids]
        pred = pd.DataFrame({"scenario": np.repeat(ids, nbs), "bus": np.concatenate([np.arange(n) for n in nbs]),
                             "vm": np.concatenate([preds[int(s)][0] for s in ids]), "va": np.concatenate([preds[int(s)][1] for s in ids])})
    pred = pred.sort_values(["scenario", "bus"]).reset_index(drop=True)
    A = load_chunk(find_local(LOCAL_DIR, level, "bus_data"), find_local(LOCAL_DIR, level, "branch_data"), pred, ids)
    return {int(ids[i]): extract_scenario(A, i) for i in range(len(ids))}


EV20 = truth_dicts(20, SEL[20], GENCO_OUT["fp32"])
BASE = truth_dicts(BASE_LEVEL, BASE_IDS)
print("truth loaded:", len(EV20), "N-20 scenarios,", len(BASE), "base points")


def adaptive(ev, vm, va_deg):
    # Newton steps until the mismatch is below TOL_SCREEN (at least one); returns steps and metrics after
    tr = newton_trace(ev.Y, ev.Sspec, ev.state(vm, va_deg), ev.pv, ev.pq, tol=TOL, max_it=MAX_NR)
    j = next((j for j in range(1, len(tr)) if tr[j][1] <= TOL_SCREEN), len(tr) - 1)
    V = tr[j][0]
    return j, ev.metrics(np.abs(V), np.degrees(np.angle(V))), V

## A. Professional solver: lightsim2grid (C++, KLU) against our Python Newton-Raphson

Same machine, same N-20 scenarios. The check uses exactly the number of Newton steps that the adaptive rule needs for
each scenario and start.

In [ ]:
rows = []
for s in list(EV20)[:N_TIMING]:
    d = EV20[s]; ev = ScenarioEval(d)
    Y, S, ref, pv, pq = system_of(d)
    starts = {"GENCO fp16c": GENCO_OUT["fp16c"][s], "GENCO fp32": GENCO_OUT["fp32"][s]}
    dc_V = ev.state(np.ones(d["nb"]), np.nan_to_num(d["Va_dc"]))
    flat_V = ev.state(np.ones(d["nb"]), np.zeros(d["nb"]))
    j_dc, _, _ = adaptive(ev, np.abs(dc_V), np.degrees(np.angle(dc_V)))
    r = {"scenario": s, "steps DC": j_dc}
    for name, (vm, va) in starts.items():
        r[f"steps {name}"] = adaptive(ev, vm, np.degrees(va))[0]
    # DC power flow: factor and solve of the reduced susceptance matrix (matrix built beforehand, like the Jacobian)
    keep = np.setdiff1d(np.arange(d["nb"]), ref)
    on = d["on"]; b = d["Yft"][on].imag
    B = sp.csr_matrix((np.r_[b, b, -b, -b], (np.r_[d["f"][on], d["t"][on], d["f"][on], d["t"][on]],
                                             np.r_[d["f"][on], d["t"][on], d["t"][on], d["f"][on]])), shape=(d["nb"], d["nb"])).tocsc()[keep][:, keep].tocsc()
    P = ((d["Pg"] - d["Pd"]) / SN_MVA)[keep]
    t0 = time.perf_counter(); splu(B).solve(P); r["DC solve ms"] = 1000 * (time.perf_counter() - t0)
    if HAVE_KLU:
        r["KLU full AC from DC start ms"] = 1000 * klu_solve(Y, S, dc_V, ref, pv, pq, 30, TOL)[3]
        r["KLU full AC from flat ms"] = 1000 * klu_solve(Y, S, flat_V, ref, pv, pq, 30, TOL)[3]
        r["KLU iterations from DC"] = klu_solve(Y, S, dc_V, ref, pv, pq, 30, TOL)[2]
        r["KLU check from DC ms"] = 1000 * klu_solve(Y, S, dc_V, ref, pv, pq, j_dc, 1e-14)[3]
        vm, va = GENCO_OUT["fp16c"][s]
        r["KLU check from GENCO ms"] = 1000 * klu_solve(Y, S, ev.state(vm, np.degrees(va)), ref, pv, pq, r["steps GENCO fp16c"], 1e-14)[3]
    t0 = time.perf_counter(); newton(ev.Y, ev.Sspec, dc_V, ev.pv, ev.pq, tol=TOL, max_it=30); r["Python full AC from DC ms"] = 1000 * (time.perf_counter() - t0)
    t0 = time.perf_counter(); newton(ev.Y, ev.Sspec, dc_V, ev.pv, ev.pq, n_steps=1); r["Python one step ms"] = 1000 * (time.perf_counter() - t0)
    rows.append(r)
SOLVER = pd.DataFrame(rows)
SOLVER.to_csv(Path(OUT_DIR) / "solver_timing.csv", index=False)
med = SOLVER.median(numeric_only=True)
display(med.drop("scenario").to_frame("median"))

cost = []
gf, g32 = T_GENCO["fp16c"], T_GENCO["fp32"]
for solver, full, chk_dc, chk_g in [("Python NR (ours)", med["Python full AC from DC ms"], med["DC solve ms"] + SOLVER["steps DC"].mean() * med["Python one step ms"],
                                     SOLVER["steps GENCO fp16c"].mean() * med["Python one step ms"])] + \
        ([("lightsim2grid KLU", med["KLU full AC from DC start ms"], med["DC solve ms"] + SOLVER["KLU check from DC ms"].mean(),
           SOLVER["KLU check from GENCO ms"].mean())] if HAVE_KLU else []):
    cost += [{"solver": solver, "workflow": "full AC solve from DC start", "ms per scenario": full},
             {"solver": solver, "workflow": "DC power flow + check", "ms per scenario": chk_dc},
             {"solver": solver, "workflow": "GENCO compiled float16 + check", "ms per scenario": gf + chk_g},
             {"solver": solver, "workflow": "GENCO float32 + check", "ms per scenario": g32 + chk_g},
             {"solver": solver, "workflow": "break-even GENCO time", "ms per scenario": chk_dc - chk_g}]
COST = pd.DataFrame(cost)
COST.to_csv(Path(OUT_DIR) / "cost.csv", index=False)
display(COST.pivot(index="workflow", columns="solver", values="ms per scenario"))

## B. Compiled float16 GENCO followed by the Newton check (N-20)

In [ ]:
rows = []
for s, d in EV20.items():
    ev = ScenarioEval(d)
    for name in ["fp32", "fp16c"]:
        vm, va = GENCO_OUT[name][s]
        m0 = ev.metrics(vm, np.degrees(va))
        j, m1, _ = adaptive(ev, vm, np.degrees(va))
        rows.append({"scenario": s, "sample": "hard case" if s in HARD20 else "random", "GENCO": name,
                     "worst Vm error": m0["err_vm_max"], "missed before": m0["missed_overloads"] + m0["missed_vviol"],
                     "newton steps": j, "missed after": m1["missed_overloads"] + m1["missed_vviol"]})
B_RES = pd.DataFrame(rows)
B_RES.to_csv(Path(OUT_DIR) / "fp16c_check.csv", index=False)
summ = B_RES.groupby(["sample", "GENCO"]).agg(scenarios=("scenario", "count"), median_worst_Vm_error=("worst Vm error", "median"),
                                             missed_per_100_before=("missed before", lambda x: 100 * x.sum() / len(x)),
                                             mean_newton_steps=("newton steps", "mean"),
                                             missed_per_100_after=("missed after", lambda x: 100 * x.sum() / len(x)))
display(summ)

## C. Storm-like outages: lines removed in one area, against lines removed anywhere

For each base operating point and each total outage level, two new cases: **clustered** (the lines nearest to a random
centre bus) and **uniform** (random lines anywhere). Lines whose removal would split the grid are skipped. Each case is
solved exactly (lightsim2grid, starting from the base solution, then flat, then DC). Cases for which none of the tested Newton starts converges are
counted, not used.

In [ ]:
STORMS, FAILS = [], []
sid = 900_000_000
rng = np.random.default_rng(SEED + 1)
for b_id in BASE_IDS:
    d = BASE[int(b_id)]
    for T in STORM_TOTALS:
        extra = T - BASE_LEVEL
        if extra <= 0:
            continue
        for mode in ["clustered", "uniform"]:
            rem, radius = pick_outages(d, extra, mode, rng)
            if rem is None:
                FAILS.append({"base": int(b_id), "T": T, "mode": mode, "reason": "could not pick lines"}); continue
            d2 = solve_new_topology(d, rem, tol=TOL)
            if d2 is None:
                FAILS.append({"base": int(b_id), "T": T, "mode": mode, "reason": "no AC solution"}); continue
            sid += 1
            STORMS.append({"sid": sid, "base": int(b_id), "T": T, "mode": mode, "radius_hops": radius,
                           "removed_lines": d["line"][rem].astype(int).tolist(), "d": d2})
FAILS = pd.DataFrame(FAILS)
print(f"storm-like cases solved: {len(STORMS)} | not usable: {len(FAILS)}")
if len(FAILS):
    display(FAILS.groupby(["mode", "T", "reason"]).size().rename("cases").to_frame())

In [ ]:
# [GPU] build storm graphs and run GENCO

bus = read_all(find_local(LOCAL_DIR, BASE_LEVEL, "bus_data"), BASE_IDS)
gen = read_all(find_local(LOCAL_DIR, BASE_LEVEL, "gen_data"), BASE_IDS)
br = read_all(find_local(LOCAL_DIR, BASE_LEVEL, "branch_data"), BASE_IDS)
bus = merge_agg_gen_into_bus(bus, gen)
bg, gg, rg = bus.groupby("scenario"), gen.groupby("scenario"), br.groupby("scenario")
STORM_GRAPHS = []
for c_ in STORMS:
    b = bg.get_group(c_["base"]).sort_values("bus").reset_index(drop=True)
    r_ = rg.get_group(c_["base"]).copy()
    r_.loc[r_["idx"].astype(int).isin(c_["removed_lines"]), "br_status"] = 0.0
    data = build_hetero_data_for_scenario(int(c_["sid"]), b, gg.get_group(c_["base"]), r_)
    NORMALIZER.transform(data)
    STORM_GRAPHS.append(TRANSFORM(data))
STORM_OUT = {"fp32": run_genco(STORM_GRAPHS, MODEL, False), "fp16c": run_genco(STORM_GRAPHS, CMODEL, True)}
print("GENCO run on", len(STORM_GRAPHS), "storm-like cases")

In [ ]:
rows = []
for c_ in STORMS:
    d2 = c_["d"]; ev = ScenarioEval(d2); s = c_["sid"]
    vm, va = STORM_OUT["fp16c"][s]
    vm32, va32 = STORM_OUT["fp32"][s]
    m0 = ev.metrics(vm32, np.degrees(va32))
    jg, mg, _ = adaptive(ev, vm, np.degrees(va))
    dcV = ev.state(np.ones(d2["nb"]), d2["Va_dc"])
    jd, md_, _ = adaptive(ev, np.abs(dcV), np.degrees(np.angle(dcV)))
    rows.append({"sid": s, "base": c_["base"], "T": c_["T"], "mode": c_["mode"], "radius_hops": c_["radius_hops"],
                 "true overloads": int(ev.over_t.sum()), "true V violations": int(ev.vio_t.sum()),
                 "GENCO worst Vm error": m0["err_vm_max"], "GENCO hard case": m0["err_vm_max"] > BAD_VM,
                 "GENCO missed (no check)": m0["missed_overloads"] + m0["missed_vviol"],
                 "GENCO steps": jg, "GENCO missed after check": mg["missed_overloads"] + mg["missed_vviol"],
                 "DC steps": jd, "DC missed after check": md_["missed_overloads"] + md_["missed_vviol"]})
STORM_RES = pd.DataFrame(rows)
STORM_RES.to_csv(Path(OUT_DIR) / "storm_cases.csv", index=False)
agg = STORM_RES.groupby(["T", "mode"]).agg(
    cases=("sid", "count"), true_overloads_per_case=("true overloads", "mean"), true_V_violations_per_case=("true V violations", "mean"),
    median_GENCO_worst_Vm_error=("GENCO worst Vm error", "median"), GENCO_hard_cases_pct=("GENCO hard case", lambda x: 100 * x.mean()),
    GENCO_missed_per_100_no_check=("GENCO missed (no check)", lambda x: 100 * x.sum() / len(x)),
    GENCO_check_steps=("GENCO steps", "mean"), GENCO_missed_per_100_after=("GENCO missed after check", lambda x: 100 * x.sum() / len(x)),
    DC_check_steps=("DC steps", "mean"), DC_missed_per_100_after=("DC missed after check", lambda x: 100 * x.sum() / len(x)))
agg.to_csv(Path(OUT_DIR) / "storm_summary.csv")
display(agg.T)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
labels = [f"N-{T}\n{m}" for T, m in agg.index]
x = np.arange(len(agg)); cols = ["#e07b20" if m == "clustered" else "#1f5fa8" for _, m in agg.index]
axes[0].bar(x, agg["median_GENCO_worst_Vm_error"], color=cols); axes[0].set_title("GENCO median worst voltage error (p.u.)")
axes[1].bar(x, agg["GENCO_hard_cases_pct"], color=cols); axes[1].set_title(f"cases with GENCO error > {BAD_VM} p.u. (%)")
axes[2].bar(x - 0.2, agg["GENCO_missed_per_100_no_check"], 0.4, color=cols, alpha=0.5, label="GENCO alone")
axes[2].bar(x + 0.2, agg["GENCO_missed_per_100_after"], 0.4, color=cols, label="GENCO + Newton check")
axes[2].set_title("missed decisions per 100 cases"); axes[2].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.set_xticks(x); ax.set_xticklabels(labels); ax.grid(alpha=0.3, axis="y")
fig.suptitle("Storm-like (clustered) against random (uniform) outages, same operating points")
fig.tight_layout(); fig.savefig(Path(OUT_DIR) / "fig16_storms.png", dpi=150); plt.show()

## D. Example cases for the interactive demo

In [ ]:
def full_loading(d, V):
    Vf, Vt = V[d["f"]], V[d["t"]]
    Sf = Vf * np.conj(d["Yff"] * Vf + d["Yft"] * Vt) * SN_MVA
    St = Vt * np.conj(d["Ytt"] * Vt + d["Ytf"] * Vf) * SN_MVA
    ld = np.maximum(np.abs(Sf), np.abs(St)) / np.where(d["rate"] > 0, d["rate"], np.nan)
    return np.where(d["on"], ld, np.nan)


def demo_case(name, kind, d, vm, va):
    ev = ScenarioEval(d)
    Vg = ev.state(vm, np.degrees(va))
    V1, _, _, _ = newton(ev.Y, ev.Sspec, Vg, ev.pv, ev.pq, n_steps=1)
    Vt = d["Vm"] * np.exp(1j * np.radians(d["Va"]))
    r4 = lambda a: [None if not np.isfinite(v) else round(float(v), 4) for v in a]
    return {"name": name, "kind": kind, "lines_out": [int(i) for i in np.nonzero(~d["on"])[0]],
            "vm_true": r4(np.abs(Vt)), "vm_genco": r4(np.abs(Vg)), "vm_check": r4(np.abs(V1)),
            "ld_true": r4(full_loading(d, Vt)), "ld_genco": r4(full_loading(d, Vg)), "ld_check": r4(full_loading(d, V1)),
            "metrics_genco": ev.metrics(np.abs(Vg), np.degrees(np.angle(Vg))), "metrics_check": ev.metrics(np.abs(V1), np.degrees(np.angle(V1)))}


demo = []
rnd_ids = [s for s in EV20 if s not in HARD20][: N_DEMO["random"]]
hard_ids = [s for s in EV20 if s in HARD20][: N_DEMO["hard"]]
for s in rnd_ids + hard_ids:
    vm, va = GENCO_OUT["fp16c"][s]
    demo.append(demo_case(f"N-20 scenario {s}", "hard" if s in HARD20 else "random", EV20[s], vm, va))
for mode in ["clustered", "uniform"]:
    for c_ in [c for c in STORMS if c["mode"] == mode and c["T"] == max(STORM_TOTALS)][: N_DEMO[mode]]:
        vm, va = STORM_OUT["fp16c"][c_["sid"]]
        demo.append(demo_case(f"storm-like N-{c_['T']} ({mode}) case {c_['sid']}", mode, c_["d"], vm, va))
d0 = next(iter(EV20.values()))
grid = {"n_bus": int(d0["nb"]), "f": d0["f"].astype(int).tolist(), "t": d0["t"].astype(int).tolist(),
        "rate": [round(float(x), 2) for x in d0["rate"]], "vmin": [round(float(x), 3) for x in d0["min_vm_pu"]],
        "vmax": [round(float(x), 3) for x in d0["max_vm_pu"]], "kv": [round(float(x), 1) for x in d0.get("vn_kv", np.zeros(d0["nb"]))],
        "type": ["REF" if r else "PV" if p else "PQ" for r, p in zip(d0["REF"], d0["PV"])]}
(Path(OUT_DIR) / "demo_cases.json").write_text(json.dumps({"grid": grid, "cases": demo}, default=float))
print(f"saved {len(demo)} demo cases ({(Path(OUT_DIR) / 'demo_cases.json').stat().st_size / 1e6:.1f} MB)")

## Output files

In [ ]:
summary = {"config": {"N_NORMAL": N_NORMAL, "N_BASE": N_BASE, "STORM_TOTALS": STORM_TOTALS, "TOL_SCREEN": TOL_SCREEN, "SEED": SEED},
           "gpu": str(globals().get("DEVICE", "unknown")), "compiled": bool(globals().get("COMPILED", False)),
           "lightsim2grid": bool(HAVE_KLU), "genco_ms": globals().get("T_GENCO", {}),
           "torch_scatter_replacement": bool(globals().get("USED_SHIM", False))}
(Path(OUT_DIR) / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
zip_path = Path(OUT_DIR).with_suffix(".zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(OUT_DIR).iterdir()):
        zf.write(p, arcname=f"{Path(OUT_DIR).name}/{p.name}")
print(f"Saved {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB).")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass